# Licence Plate Detection Pipeline

Version: 1.0.1

Key Features:
- Voting based OCR for improved accuracy

Mount the drive to access the video file and the model

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Path to the video file and model:
You can specify the path to your video file and the output path for the processed video.

In [2]:
# Import datetime for timestamping output video files
from datetime import datetime
# Import Path from pathlib for handling file paths
from pathlib import Path

VIDEO_SOURCE: str = (
    r"/content/drive/MyDrive/DATA/Input_Videos/IMG_1552.MOV"
)
# Temporarily change output path to /content/ to diagnose Google Drive write issues
OUTPUT_VIDEO_PATH: str = (
    f"/content/drive/MyDrive/DATA/Output_Videos/temp_output_{datetime.now().strftime('%Y%m%d%H%M%S')}.mp4"
)

# YOLO Model Path
MODEL_PATH = Path(
    r"/content/drive/MyDrive/DATA/Models/best_license_plate_model.pt"
)

Import necessary libraries for the pipeline

In [3]:
!pip uninstall opencv-python-headless opencv-python -y
# !pip install ultralytics==8.3.152 easyocr==1.7.2 transformers==4.52.4 opencv-python pyyaml>=6.0

Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93


In [4]:
!pip install ultralytics easyocr==1.7.2 transformers==4.52.4 pyyaml>=6.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
!pip install opencv-python==4.11.0.86 opencv-python-headless==4.12.0.88

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 11.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 5.0.0.93
    Uninstalling opencv-python-5.0.0.93:
      Successfully uninstalled opencv-python-5.0.0.93


In [6]:
import os
import cv2
print(cv2.__version__)
import sys
import time
import torch
import numpy as np
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from PIL import Image
from datetime import datetime
from threading import Thread
from queue import Queue
from collections import Counter
from dataclasses import dataclass, field
from ultralytics import YOLO
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

4.11.0
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Other necessary configurations and parameters for the pipeline

In [7]:
# For faster computation, use FP16
FP16_ENABLED = True
# Resize dimensions
RESIZE_DIMENSIONS: Tuple[int, int] = (1280, 720)
# How long to keep plates visible (in frames)
PLATE_PERSISTENCE_FRAMES: int = 150
# Minimum frames between same plate detections to consider it "new"
UNIQUE_DETECTION_COOLDOWN: int = 30
# Show/hide the persistent detection panel (True = show, False = hide)
SHOW_DETECTION_PANEL: bool = True

# ============== QUALITY ASSESSMENT THRESHOLDS ==============
# Minimum blur score (Laplacian variance) - higher = sharper
MIN_BLUR_SCORE: float = 100.0
# Maximum percentage of blown-out pixels (pure white/black)
MAX_BLOWN_OUT_RATIO: float = 0.15
# Minimum plate dimensions (width x height in pixels)
MIN_PLATE_WIDTH: int = 100
MIN_PLATE_HEIGHT: int = 40
# Number of best frames to keep per tracked plate
BEST_FRAMES_BUFFER_SIZE: int = 5
# Frames without detection before triggering OCR (plate "left" the frame)
PLATE_EXIT_THRESHOLD: int = 15
# Maximum frames to wait before running OCR even if plate is still visible
MAX_TRACKING_FRAMES: int = 90  # ~3 seconds at 30fps

Dataclass for storing the candidate license plate information

In [8]:
@dataclass
class PlateCandidate:
    """Stores a candidate plate crop with its quality metrics."""

    frame_num: int
    crop: np.ndarray
    bbox: Tuple[int, int, int, int]
    confidence: float
    blur_score: float
    exposure_score: float
    quality_score: float  # Combined quality metric

Dataclass to store tracked plate information across frames

In [9]:
@dataclass
class TrackedPlate:
    """Stores tracking info and best frame buffer for a tracked plate."""

    track_id: int
    first_seen: int
    last_seen: int
    best_candidates: List[PlateCandidate] = field(default_factory=list)
    final_ocr_text: Optional[str] = None
    ocr_completed: bool = False
    vote_details: Optional[Dict] = None

Threaded video reader for efficient frame capture

In [10]:
class ThreadedVideoReader:
    """Reads frames in a background thread so the GPU never waits for the hard drive."""

    def __init__(self, path, queue_size=128):
        self.stream = cv2.VideoCapture(path)
        self.stopped = False
        self.Q = Queue(maxsize=queue_size)
        self.width = int(self.stream.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.stream.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = int(self.stream.get(cv2.CAP_PROP_FPS))
        self.total_frames = int(self.stream.get(cv2.CAP_PROP_FRAME_COUNT))

    def start(self):
        t = Thread(target=self.update, args=())
        t.daemon = True
        t.start()
        return self

    def update(self):
        while True:
            if self.stopped:
                return
            if not self.Q.full():
                ret, frame = self.stream.read()
                if not ret:
                    self.stopped = True
                    return
                self.Q.put(frame)
            else:
                time.sleep(0.01)

    def read(self):
        return self.Q.get() if not self.Q.empty() else None

    def more(self):
        return not (self.stopped and self.Q.empty())

    def release(self):
        self.stopped = True
        self.stream.release()

Core Video Processing Loop

In [11]:
class LicensePlateVideoAnalyzer:
    def __init__(self):

        # Setup GPU if available
        self._setup_gpu()

        # Load models
        self._load_models()

        # Initialize tracking
        self.detections = []
        self.tracked_plates: Dict[int, TrackedPlate] = {}  # Track plates by ID
        self.recent_detections = []  # Recently detected plates for display
        self.finalized_plates: Dict[int, str] = {}  # Plates that completed OCR

        print("LicensePlateVideoAnalyzer initialized successfully.")
        print(
            f"Quality thresholds: blur>{MIN_BLUR_SCORE}, blown_out<{MAX_BLOWN_OUT_RATIO*100}%, min_size={MIN_PLATE_WIDTH}x{MIN_PLATE_HEIGHT}"
        )
        print(
            f"Buffer size: {BEST_FRAMES_BUFFER_SIZE} frames, exit threshold: {PLATE_EXIT_THRESHOLD} frames"
        )

    def _setup_gpu(self):
        self.device = "cpu"
        self.use_gpu = False

        if torch.cuda.is_available():
            self.device = "cuda"
            self.use_gpu = True

            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f"Using GPU: {gpu_name} with {gpu_memory:.2f} GB memory")
        else:
            print("Using CPU for processing")

    def _load_models(self):
        """Load YOLO and TrOCR models."""

        model_path = Path(MODEL_PATH)

        if not model_path.exists():
            print(f"Model file not found at {model_path}")
            sys.exit(1)

        print(f"Loading YOLO model from {model_path}")
        self.yolo_model = YOLO(str(model_path))

        if self.use_gpu:
            self.yolo_model.to(self.device)

        print(f" YOLO model loaded (classes: {self.yolo_model.names})")

        print("Loading TrOCR model and processor")
        trocr_model_name = "microsoft/trocr-base-printed"
        print(f"Loading TrOCR model: {trocr_model_name}")
        self.processor = TrOCRProcessor.from_pretrained(trocr_model_name, use_fast=True)
        print(f"Loading TrOCR vision encoder model")
        self.trocr_model = VisionEncoderDecoderModel.from_pretrained(trocr_model_name)

        # Move TrOCR model to GPU
        if self.use_gpu:
            self.trocr_model.to(self.device)

            # Use FP16 if enabled
            if FP16_ENABLED:
                self.trocr_model.half()

        print("TrOCR model and processor loaded successfully")

    def _calculate_blur_score(self, image: np.ndarray) -> float:
        """
        Calculate blur score using Laplacian variance.
        Higher values = sharper image, lower values = blurry.
        """
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image

        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        variance = laplacian.var()
        return float(variance)

    def _calculate_exposure_score(self, image: np.ndarray) -> Tuple[float, bool]:
        """
        Check if image is properly exposed by analyzing histogram.
        Returns (blown_out_ratio, is_acceptable).
        """
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image

        total_pixels = gray.size

        # Count pure black (0-5) and pure white (250-255) pixels
        black_pixels = np.sum(gray <= 5)
        white_pixels = np.sum(gray >= 250)
        blown_out_ratio = (black_pixels + white_pixels) / total_pixels

        is_acceptable = blown_out_ratio < MAX_BLOWN_OUT_RATIO
        return float(blown_out_ratio), is_acceptable

    def _check_size(self, width: int, height: int) -> bool:
        """Check if plate crop meets minimum size requirements."""
        return width >= MIN_PLATE_WIDTH and height >= MIN_PLATE_HEIGHT

    def _assess_quality(
        self, crop: np.ndarray, bbox: Tuple[int, int, int, int]
    ) -> Tuple[float, bool, Dict]:
        """
        Comprehensive quality assessment for a plate crop.
        Returns (quality_score, passes_filter, metrics_dict).
        """
        x1, y1, x2, y2 = bbox
        width = x2 - x1
        height = y2 - y1

        # Size check
        size_ok = self._check_size(width, height)
        if not size_ok:
            return 0.0, False, {"reason": "too_small", "width": width, "height": height}

        # Blur check
        blur_score = self._calculate_blur_score(crop)
        blur_ok = blur_score >= MIN_BLUR_SCORE

        # Exposure check
        blown_out_ratio, exposure_ok = self._calculate_exposure_score(crop)

        # Calculate combined quality score (weighted)
        # Normalize blur score (cap at 1000 for scoring)
        normalized_blur = min(blur_score / 1000.0, 1.0)
        # Invert blown_out_ratio (lower is better)
        exposure_score = 1.0 - blown_out_ratio
        # Size bonus (larger plates get higher scores)
        size_score = min((width * height) / (200 * 80), 1.0)

        # Weighted combination
        quality_score = (
            (normalized_blur * 0.5) + (exposure_score * 0.3) + (size_score * 0.2)
        )

        passes_filter = size_ok and blur_ok and exposure_ok

        metrics = {
            "blur_score": blur_score,
            "blur_ok": blur_ok,
            "blown_out_ratio": blown_out_ratio,
            "exposure_ok": exposure_ok,
            "size": (width, height),
            "size_ok": size_ok,
            "quality_score": quality_score,
            "passes_filter": passes_filter,
        }

        return quality_score, passes_filter, metrics

    def _update_best_candidates(
        self, tracked_plate: TrackedPlate, candidate: PlateCandidate
    ):
        """
        Update the best candidates buffer for a tracked plate.
        Keeps only the top BEST_FRAMES_BUFFER_SIZE highest quality candidates.
        """
        candidates = tracked_plate.best_candidates
        candidates.append(candidate)

        # Sort by quality score (descending) and keep only the best
        candidates.sort(key=lambda c: c.quality_score, reverse=True)
        tracked_plate.best_candidates = candidates[:BEST_FRAMES_BUFFER_SIZE]

    def _run_majority_voting_ocr(self, tracked_plate: TrackedPlate) -> Tuple[str, Dict]:
        """
        Run OCR on best candidate frames and use majority voting.
        Returns (final_text, voting_details).
        """
        candidates = tracked_plate.best_candidates

        if not candidates:
            return "NO_CANDIDATES", {"error": "No candidates available"}

        ocr_results = []

        # Run OCR on each candidate
        for candidate in candidates:
            text = self._run_ocr(candidate.crop)
            if text and text != "OCR_ERROR":
                ocr_results.append(text)

        if not ocr_results:
            return "OCR_FAILED", {"error": "All OCR attempts failed"}

        # Simple majority voting - count exact matches
        text_counts = Counter(ocr_results)
        most_common = text_counts.most_common()

        # If we have a clear winner (appears more than once), use it
        if most_common[0][1] > 1:
            final_text = most_common[0][0]
            voting_details = {
                "method": "exact_match",
                "winner": final_text,
                "count": most_common[0][1],
                "total_votes": len(ocr_results),
                "all_results": ocr_results,
            }
            return final_text, voting_details

        # No exact match - try character-level voting
        if len(ocr_results) >= 2:
            final_text = self._character_level_voting(ocr_results)
            voting_details = {
                "method": "character_voting",
                "winner": final_text,
                "all_results": ocr_results,
            }
            return final_text, voting_details

        # Fallback to first result
        final_text = ocr_results[0]
        voting_details = {
            "method": "single_result",
            "winner": final_text,
            "all_results": ocr_results,
        }
        return final_text, voting_details

    def _character_level_voting(self, ocr_results: List[str]) -> str:
        """
        Perform character-level voting when exact matches fail.
        Aligns strings and votes on each character position.
        """
        if not ocr_results:
            return ""

        # Normalize: remove spaces and convert to uppercase
        normalized = [text.replace(" ", "").upper() for text in ocr_results]

        # Find the most common length
        lengths = Counter(len(s) for s in normalized)
        target_length = lengths.most_common(1)[0][0]

        # Filter to strings of the target length
        same_length = [s for s in normalized if len(s) == target_length]

        if not same_length:
            # Fallback: use the longest string
            return max(normalized, key=len)

        # Vote character by character
        result = []
        for i in range(target_length):
            chars = [s[i] for s in same_length]
            char_counts = Counter(chars)
            winner = char_counts.most_common(1)[0][0]
            result.append(winner)

        return "".join(result)

    def _check_plates_for_ocr(self, current_frame: int) -> List[int]:
        """
        Check which tracked plates should now run OCR.
        Triggers when: plate exits frame OR max tracking time exceeded.
        Returns list of track_ids that need OCR.
        """
        plates_to_process = []

        for track_id, tracked_plate in self.tracked_plates.items():
            if tracked_plate.ocr_completed:
                continue

            frames_since_seen = current_frame - tracked_plate.last_seen
            total_tracking_frames = current_frame - tracked_plate.first_seen

            # Trigger conditions
            plate_exited = frames_since_seen >= PLATE_EXIT_THRESHOLD
            max_time_exceeded = total_tracking_frames >= MAX_TRACKING_FRAMES
            has_candidates = len(tracked_plate.best_candidates) > 0

            if has_candidates and (plate_exited or max_time_exceeded):
                plates_to_process.append(track_id)

        return plates_to_process

    def _run_ocr(self, cropped_plate: np.ndarray) -> str:
        """Run OCR on cropped license plate"""
        try:
            if len(cropped_plate.shape) == 2:
                pil_img = Image.fromarray(cropped_plate)
            else:
                pil_img = Image.fromarray(
                    cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB)
                )

            # Process with TrOCR
            inputs = self.processor(images=pil_img, return_tensors="pt").pixel_values

            if self.use_gpu:
                inputs = inputs.to(self.device)
                if FP16_ENABLED:
                    inputs = inputs.half()

            with torch.no_grad():
                outputs = self.trocr_model.generate(inputs)

            text = self.processor.batch_decode(outputs, skip_special_tokens=True)[0]
            return text.strip()

        except Exception as e:
            return "OCR_ERROR"

    def _draw_detection_panel(self, frame: np.ndarray, frame_num: int) -> np.ndarray:
        """Draw a persistent panel showing finalized license plate detections"""
        # Update recent detections - remove old ones
        self.recent_detections = [
            det
            for det in self.recent_detections
            if frame_num - det["last_seen"] < PLATE_PERSISTENCE_FRAMES
        ]

        if not self.recent_detections:
            return frame

        # Panel configuration
        panel_width = 420
        panel_x = frame.shape[1] - panel_width - 10
        panel_y = 10
        line_height = 55

        # Draw semi-transparent background panel
        overlay = frame.copy()
        panel_height = min(
            len(self.recent_detections) * line_height + 40, frame.shape[0] - 20
        )
        cv2.rectangle(
            overlay,
            (panel_x, panel_y),
            (panel_x + panel_width, panel_y + panel_height),
            (0, 0, 0),
            -1,
        )
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

        # Draw title
        cv2.putText(
            frame,
            "FINALIZED PLATES (Voted)",
            (panel_x + 10, panel_y + 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

        # Draw each recent detection
        y_offset = panel_y + 55
        for i, det in enumerate(self.recent_detections[:8]):  # Show max 8
            plate_text = det["ocr_text"]
            conf = det["confidence"]
            age = frame_num - det["last_seen"]
            method = det.get("method", "unknown")

            # Color based on freshness (green = fresh, yellow = old)
            freshness = 1 - (age / PLATE_PERSISTENCE_FRAMES)
            color = (0, int(255 * freshness), int(255 * (1 - freshness)))

            # Draw plate text with large font
            text = f"{plate_text}"
            cv2.putText(
                frame,
                text,
                (panel_x + 15, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                color,
                2,
            )

            # Draw confidence, count, and voting method
            subtext = f"Conf: {conf:.2f} | Frames: {det['count']} | {method}"
            cv2.putText(
                frame,
                subtext,
                (panel_x + 15, y_offset + 22),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (200, 200, 200),
                1,
            )

            y_offset += line_height

        return frame

    def _process_frame(
        self, frame: np.ndarray, frame_num: int
    ) -> Tuple[np.ndarray, List[Dict]]:
        """Process single frame for license plate detection using tracking + quality buffering.

        Uses YOLO's built-in track mode to assign persistent IDs. Instead of running
        OCR on every frame, we buffer the highest quality crops and run OCR only when
        the plate exits the frame or max tracking time is reached.
        """
        ann_frame = frame.copy()
        frame_detections = []

        # Check if any tracked plates need OCR processing
        plates_to_ocr = self._check_plates_for_ocr(frame_num)
        for track_id in plates_to_ocr:
            tracked_plate = self.tracked_plates[track_id]
            ocr_text, vote_details = self._run_majority_voting_ocr(tracked_plate)
            tracked_plate.final_ocr_text = ocr_text
            tracked_plate.ocr_completed = True
            tracked_plate.vote_details = vote_details
            self.finalized_plates[track_id] = ocr_text

            # Add to recent detections for panel display
            best_conf = max(
                (c.confidence for c in tracked_plate.best_candidates), default=0.0
            )
            self.recent_detections.insert(
                0,
                {
                    "ocr_text": ocr_text,
                    "confidence": best_conf,
                    "last_seen": frame_num,
                    "count": len(tracked_plate.best_candidates),
                    "method": vote_details.get("method", "unknown"),
                },
            )

            print(
                f"[Frame {frame_num}] Finalized plate ID:{track_id} -> '{ocr_text}' (method: {vote_details.get('method')}, candidates: {len(tracked_plate.best_candidates)})"
            )

        # Use YOLO TRACKING to get persistent IDs
        results = self.yolo_model.track(
            frame, persist=True, verbose=False, tracker="bytetrack.yaml"
        )

        for result in results:
            boxes = result.boxes

            if boxes.id is None:
                continue

            box_ids = boxes.id.int().cpu().tolist()
            box_coords = boxes.xyxy.cpu().tolist()
            box_confs = boxes.conf.cpu().tolist()
            box_cls = boxes.cls.int().cpu().tolist()

            for i, track_id in enumerate(box_ids):
                x1, y1, x2, y2 = map(int, box_coords[i])
                confidence = box_confs[i]
                class_id = box_cls[i]
                detected_class = self.yolo_model.names.get(class_id, "Unknown")

                if confidence < 0.5:
                    continue

                bbox = (x1, y1, x2, y2)
                cropped_plate = frame[y1:y2, x1:x2]

                if cropped_plate.size == 0:
                    continue

                # Assess quality of this crop
                quality_score, passes_filter, metrics = self._assess_quality(
                    cropped_plate, bbox
                )

                # Initialize or update tracked plate
                is_new_track = track_id not in self.tracked_plates

                if is_new_track:
                    self.tracked_plates[track_id] = TrackedPlate(
                        track_id=track_id, first_seen=frame_num, last_seen=frame_num
                    )

                tracked_plate = self.tracked_plates[track_id]
                tracked_plate.last_seen = frame_num

                # If quality passes filter, add to candidates buffer
                if passes_filter and not tracked_plate.ocr_completed:
                    candidate = PlateCandidate(
                        frame_num=frame_num,
                        crop=cropped_plate.copy(),
                        bbox=bbox,
                        confidence=confidence,
                        blur_score=metrics["blur_score"],
                        exposure_score=1.0 - metrics["blown_out_ratio"],
                        quality_score=quality_score,
                    )
                    self._update_best_candidates(tracked_plate, candidate)

                # Get display text
                if tracked_plate.ocr_completed:
                    display_text = tracked_plate.final_ocr_text or "UNKNOWN"
                    status = "DONE"
                elif tracked_plate.best_candidates:
                    display_text = f"Buffering ({len(tracked_plate.best_candidates)}/{BEST_FRAMES_BUFFER_SIZE})"
                    status = "BUFF"
                else:
                    display_text = "Waiting for quality frame..."
                    status = "WAIT"

                # --- VISUALIZATION ---
                # Color based on status
                if tracked_plate.ocr_completed:
                    box_color = (0, 255, 0)  # Green - completed
                elif passes_filter:
                    box_color = (0, 255, 255)  # Yellow - good quality frame
                else:
                    box_color = (0, 165, 255)  # Orange - low quality frame

                box_thickness = 3 if is_new_track else 2
                cv2.rectangle(ann_frame, (x1, y1), (x2, y2), box_color, box_thickness)

                # Draw quality indicator
                quality_bar_width = int(quality_score * 50)
                cv2.rectangle(
                    ann_frame, (x1, y2 + 2), (x1 + 50, y2 + 8), (50, 50, 50), -1
                )
                cv2.rectangle(
                    ann_frame,
                    (x1, y2 + 2),
                    (x1 + quality_bar_width, y2 + 8),
                    (0, 255, 0),
                    -1,
                )

                # Draw label
                label = f"ID:{track_id} [{status}]"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.6
                thickness = 2

                (text_w, text_h), _ = cv2.getTextSize(
                    label, font, font_scale, thickness
                )

                bg_y1 = max(y1 - text_h - 10, 0)
                cv2.rectangle(
                    ann_frame, (x1, bg_y1), (x1 + text_w + 10, y1), box_color, -1
                )
                cv2.putText(
                    ann_frame,
                    label,
                    (x1 + 5, y1 - 5),
                    font,
                    font_scale,
                    (0, 0, 0),
                    thickness,
                )

                # Draw OCR text or status below
                if tracked_plate.ocr_completed:
                    cv2.putText(
                        ann_frame,
                        display_text,
                        (x1, y2 + 25),
                        font,
                        0.7,
                        (0, 255, 0),
                        2,
                    )
                else:
                    cv2.putText(
                        ann_frame,
                        display_text,
                        (x1, y2 + 25),
                        font,
                        0.5,
                        (200, 200, 200),
                        1,
                    )

                # Draw quality metrics for debugging (small text)
                if not tracked_plate.ocr_completed and 'blur_score' in metrics:
                    blur_text = f"B:{metrics['blur_score']:.0f}"
                    exp_text = f"E:{(1-metrics.get('blown_out_ratio', 0))*100:.0f}%"
                    cv2.putText(
                        ann_frame,
                        blur_text,
                        (x2 + 5, y1 + 15),
                        font,
                        0.4,
                        (150, 150, 150),
                        1,
                    )
                    cv2.putText(
                        ann_frame,
                        exp_text,
                        (x2 + 5, y1 + 30),
                        font,
                        0.4,
                        (150, 150, 150),
                        1,
                    )

                # Store detection
                detection = {
                    "frame_number": frame_num,
                    "timestamp": datetime.now().isoformat(),
                    "bbox": [x1, y1, x2, y2],
                    "confidence": confidence,
                    "ocr_text": display_text if tracked_plate.ocr_completed else None,
                    "class": detected_class,
                    "track_id": track_id,
                    "is_new": is_new_track,
                    "quality_score": quality_score,
                    "passes_filter": passes_filter,
                    "ocr_completed": tracked_plate.ocr_completed,
                }
                frame_detections.append(detection)

        # Draw the persistent detection panel (if enabled)
        if SHOW_DETECTION_PANEL:
            ann_frame = self._draw_detection_panel(ann_frame, frame_num)

        return ann_frame, frame_detections

    def process_video(
        self, video_source: str, output_path: Optional[str] = None
    ) -> str:  # type: ignore
        """
        Process video with license plate detection using threaded reading.
        """
        if not video_source:
            raise ValueError("Video source path is empty.")

        print(f"Processing video: {video_source}")

        # Use ThreadedVideoReader for better performance
        print("Starting threaded video reader...")
        reader = ThreadedVideoReader(video_source).start()

        self.fps = reader.fps
        self.frame_count = reader.total_frames
        self.width = reader.width
        self.height = reader.height

        if RESIZE_DIMENSIONS:
            self.width, self.height = RESIZE_DIMENSIONS

        self.current_frame = 0

        output_path = Path(OUTPUT_VIDEO_PATH)  # type: ignore

        if not os.path.exists(output_path.parent):
            os.makedirs(output_path.parent)

        fourcc = cv2.VideoWriter.fourcc(*"mp4v")
        writer = cv2.VideoWriter(
            str(output_path), fourcc, self.fps, (self.width, self.height)
        )

        if not writer.isOpened():
            raise ValueError(f"Failed to open video writer for output: {output_path}")

        print(f"Output video will be saved to: {output_path}")
        print(f"Video info: {self.frame_count} frames @ {self.fps} FPS")

        frame_skip = 1
        frame_num = 0
        processed_frames = 0
        self.detections = []

        # Reset tracking state for new video
        self.tracked_plates = {}
        self.finalized_plates = {}
        self.recent_detections = []

        try:
            while reader.more():
                frame = reader.read()
                if frame is None:
                    continue

                frame_num += 1

                # Skip frames if needed
                if (frame_num - 1) % frame_skip != 0:
                    continue

                # Resize frame if needed
                if RESIZE_DIMENSIONS:
                    frame = cv2.resize(frame, RESIZE_DIMENSIONS)

                annotated_frame, frame_detections = self._process_frame(
                    frame, frame_num
                )
                self.detections.extend(frame_detections)

                # Resize annotated frame to match writer dimensions
                if (
                    annotated_frame.shape[1] != self.width
                    or annotated_frame.shape[0] != self.height
                ):
                    annotated_frame = cv2.resize(
                        annotated_frame, (self.width, self.height)
                    )

                # Write annotated frame to output video
                writer.write(annotated_frame)

                processed_frames += 1

                if processed_frames % 50 == 0:
                    buffering_count = sum(
                        1
                        for p in self.tracked_plates.values()
                        if not p.ocr_completed and p.best_candidates
                    )
                    print(
                        f"Processed {processed_frames}/{self.frame_count} frames... "
                        f"(Finalized: {len(self.finalized_plates)}, Buffering: {buffering_count})"
                    )

        finally:
            # Process any remaining tracked plates that haven't been finalized
            print("Processing remaining tracked plates...")
            for track_id, tracked_plate in self.tracked_plates.items():
                if not tracked_plate.ocr_completed and tracked_plate.best_candidates:
                    ocr_text, vote_details = self._run_majority_voting_ocr(
                        tracked_plate
                    )
                    tracked_plate.final_ocr_text = ocr_text
                    tracked_plate.ocr_completed = True
                    self.finalized_plates[track_id] = ocr_text
                    print(f"  Finalized remaining plate ID:{track_id} -> '{ocr_text}'")

            reader.release()
            writer.release()

        print(
            f"Video processing completed. Processed {processed_frames} frames in total."
        )
        print(f"Unique plates finalized: {len(self.finalized_plates)}")
        print("\n=== FINAL PLATE READINGS ===")
        for track_id, text in self.finalized_plates.items():
            tracked = self.tracked_plates.get(track_id)
            if tracked and tracked.vote_details:
                method = tracked.vote_details.get("method", "unknown")
                print(f"  ID:{track_id} -> {text} (method: {method})")
            else:
                print(f"  ID:{track_id} -> {text}")

        return str(output_path)

Main function to run the license plate detection pipeline

In [12]:
def main():
    analyzer = LicensePlateVideoAnalyzer()
    output_video = analyzer.process_video(VIDEO_SOURCE, OUTPUT_VIDEO_PATH)
    print(f"Processed video saved at: {output_video}")


if __name__ == "__main__":
    main()

Using GPU: Tesla T4 with 14.56 GB memory
Loading YOLO model from /content/drive/MyDrive/DATA/Models/best_license_plate_model.pt
 YOLO model loaded (classes: {0: 'LP'})
Loading TrOCR model and processor
Loading TrOCR model: microsoft/trocr-base-printed


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TrOCR model and processor loaded successfully
LicensePlateVideoAnalyzer initialized successfully.
Quality thresholds: blur>100.0, blown_out<15.0%, min_size=100x40
Buffer size: 5 frames, exit threshold: 15 frames
Processing video: /content/drive/MyDrive/DATA/Input_Videos/IMG_1552.MOV
Starting threaded video reader...
Output video will be saved to: /content/drive/MyDrive/DATA/Output_Videos/temp_output_20260818054229.mp4
Video info: 328 frames @ 59 FPS
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 374ms
Prepared 1 package in 89ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 50/328 frames... (Finalized: 0, Buffering: 1)
[Frame 72] Finalized plate ID:9 -> 'KA04MY9530' (method: exact_match, candidates: 5)
Processed 100/328 frames... (Finalized: 1, Buffe